# Task 4.2 — EEGNet regression decoder (braindecode)

EEGNet on 1 kHz LFP predicting Index fingertip velocity per 50 ms window,
trained on stimulation-OFF trials.

- **4.2.1** Adapt braindecode's EEGNet to regression: one velocity pair per 50 ms bin.
- **4.2.2** Split the data into training days, validation days and test days.
- **4.2.3** Early stopping on the validation days.
- **4.2.4** Sweep the kernel length, dropout and depth multiplier.
- **4.2.5** Train 5 seeds with the best setting and compare held-out OFF R² with ridge (task 3.1).
- **4.2.6** Retrain with synthetic stimulation tones in the training data.
- **4.2.7** Average the 5 seeds into an ensemble.
- **4.2.8** Compare every method per day, because sessions on one day share a recording.
- **4.2.9** Separate the clean-input result from the stimulated-input result.

## Setup

In [1]:
import os
import shutil
import sys
import warnings

import pandas as pd
import torch
from braindecode.models import EEGNet

HERE = os.path.abspath(os.getcwd())
TASK4_DIR = HERE if os.path.basename(HERE) == 'task4' else os.path.join(HERE, 'task4')
if TASK4_DIR not in sys.path:
    sys.path.insert(0, TASK4_DIR)

import utils

SEEDS = (0, 1, 2, 3, 4)
FIGURE_TRAINING_LOGS = os.path.join(utils.RESULTS_DIR, 'training_logs')


### 4.2.1 CausalEEGNet

CausalEEGNet converts EEGNet into a causal, per-bin regressor by setting final_conv_length=1 and replacing symmetric padding with left-only ConstantPad2d layers, so convolutions rely strictly on current and past LFP samples.

The check below runs in `eval()` mode: in train mode BatchNorm normalises with batch
statistics, which mixes information across time positions and would make any model look
non-causal.


In [2]:
# MODELS: shared with task4/utils.py (the compat shim executes this cell)
import torch.nn as nn

MLP_DROPOUT_P = 0.3
GRU_HIDDEN_SIZE = 128

temporal_kernel_n, kernel_n = 500, 62
temporal_filters_n, spatial_filters_n, filters_n = 8, 16, 16
ave_pool1_n, ave_pool2_n = 2, 25


class MLPRegressor(nn.Module):
    """Fully connected on cached bin features: (n_bins, n_features) -> (n_bins, 2)."""

    def __init__(self, input_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 32), nn.GELU(), nn.Dropout(MLP_DROPOUT_P),
            nn.Linear(32, 16), nn.GELU(), nn.Dropout(MLP_DROPOUT_P),
            nn.Linear(16, 2),
        )

    def forward(self, bin_features):
        return self.layers(bin_features)


class GRURegressor(nn.Module):
    """Per-bin velocity from a trial's sequence of cached features.

    Input (n_trials, n_bins, n_features); output (n_trials, n_bins, 2).
    """

    def __init__(self, input_size, hidden=GRU_HIDDEN_SIZE, layers=1, dropout=0.0):
        super().__init__()
        self.gru = nn.GRU(
                input_size, 
                hidden, 
                num_layers=layers, 
                batch_first=True,
                dropout=dropout if layers > 1 else 0.0
                )
        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden, 2)

    def forward(self, trial_bins_by_features):
        hidden_states, _ = self.gru(trial_bins_by_features)
        return self.output(self.dropout(hidden_states))


class CausalEEGNet(nn.Module):
    """braindecode EEGNet with left-only padding: one (vx, vy) per 50 ms bin.

    Every padded convolution is rebuilt as a left-only pad plus an unpadded convolution,
    so a bin can only see samples up to its own end.
    """

    def __init__(self, channel_count=35, samples=2000, outputs=2, temporal_filters_n=8,
                 depth_multiplier=2, kernel_samples=500, separable_samples=62,
                 pool_first=2, pool_second=25, dropout=0.5):
        super().__init__()

        self.bin_count = samples // (pool_first * pool_second)
        if self.bin_count * pool_first * pool_second != samples:
            raise ValueError("pool_first * pool_second must divide samples into whole bins")

        backbone = EEGNet(
                n_chans=channel_count, 
                n_outputs=outputs, 
                n_times=samples,
                final_conv_length=1, 
                F1=temporal_filters_n, 
                D=depth_multiplier,
                kernel_length=kernel_samples,
                depthwise_kernel_length=separable_samples,
                pool1_kernel_size=pool_first, 
                pool2_kernel_size=pool_second,
                drop_prob=dropout
                )

        layers = []
        for module in backbone.children():
            if isinstance(module, nn.Conv2d) and module.padding[1] > 0:
                layers.append(nn.ConstantPad2d((module.kernel_size[1] - 1, 0, 0, 0), 0.0))
                module.padding = (module.padding[0], 0)
            layers.append(module)
        self.layers = nn.Sequential(*layers)

    def forward(self, trial_channels_by_samples):
        """(n_trials, n_channels, n_samples) -> (n_trials, n_bins, 2)."""
        return self.layers(trial_channels_by_samples).transpose(1, 2)


In [3]:
print(CausalEEGNet())


CausalEEGNet(
  (layers): Sequential(
    (0): Ensure4d()
    (1): Rearrange('batch ch t 1 -> batch 1 ch t')
    (2): ConstantPad2d(padding=(499, 0, 0, 0), value=0.0)
    (3): Conv2d(1, 8, kernel_size=(1, 500), stride=(1, 1), bias=False)
    (4): BatchNorm2d(8, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (5): ParametrizedConv2dWithConstraint(
      8, 16, kernel_size=(35, 1), stride=(1, 1), groups=8, bias=False
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): MaxNormParametrize()
        )
      )
    )
    (6): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
    (7): ELU(alpha=1.0)
    (8): AvgPool2d(kernel_size=(1, 2), stride=(1, 2), padding=0)
    (9): Dropout(p=0.5, inplace=False)
    (10): ConstantPad2d(padding=(61, 0, 0, 0), value=0.0)
    (11): Conv2d(16, 16, kernel_size=(1, 62), stride=(1, 1), groups=16, bias=False)
    (12): Conv2d(16, 16, kernel_size=(1, 

In [4]:
# Causality check: perturb the last input sample; only the last bin may move.
probe = CausalEEGNet().eval()
with torch.no_grad():
    silent_input = torch.zeros(1, 35, 2000)
    silent_output = probe(silent_input)
    perturbed_input = silent_input.clone()
    perturbed_input[:, :, -1] = 50.0
    bin_change = (probe(perturbed_input) - silent_output).abs().sum(dim=(0, 2))
moved_bins = (bin_change > 1e-6).nonzero().flatten().tolist()
print('bins affected by the final sample:', moved_bins)
assert moved_bins == [39], moved_bins


bins affected by the final sample: [39]


### 4.2.2 Training days, validation days and test days

The split is by *day*, never by trial, so nothing leaks across recordings. `TRAIN` is every
stimulation-off training session; `VALIDATION` the three stimulation-off validation days
from `splits.json`, which early stopping and the sweep both watch; `TEST` pairs each
stimulated test session with the same day's stimulation-off recording — that pairing is
what produces the clean-input score beside the stimulated-input score later. Five test days
carry the fourteen stimulated sessions, so a day counts once when methods are compared.

In [6]:
# Trial tensors, one row per trial. Cached features: (trials, 40 bins, 280 features).
# LFP: (trials, 35 channels, 2000 samples). Both decode to (trials, 40 bins, 2).
STIMULATION_OFF_TRAIN_DATA = utils.stimulation_off_train_dataset()
STIMULATION_OFF_VALIDATION_DATA = utils.stimulation_off_validation_dataset()
STIMULATION_ON_TEST_PAIRS = utils.stimulation_on_test_pairs()

print('training trials  ', STIMULATION_OFF_TRAIN_DATA['features'].shape)
print('validation trials', STIMULATION_OFF_VALIDATION_DATA['features'].shape)
print('test sessions    ', len(STIMULATION_ON_TEST_PAIRS),
      '| days', len({session.split('-')[1] for session in STIMULATION_ON_TEST_PAIRS}))

training trials   (560, 35, 2000)
validation trials (150, 35, 2000)
test sessions     14 | days 5


### 4.2.3 Early stopping on the validation days

`train_with_early_stopping` keeps the weights from the epoch with the best validation R²
and stops once validation stops improving, so each run spends as many epochs as it needs.
The loss is NaN-masked: bins without kinematics simply do not contribute.

In [7]:
def train_seeds(train_trials, setting, method, seeds=SEEDS):
    models, seed_summaries, score_rows = [], [], []
    for seed in seeds:
        torch.manual_seed(seed)  # train_with_early_stopping seeds after construction
        model, history = utils.train_with_early_stopping(
            CausalEEGNet(**setting),
            train_trials,
            STIMULATION_OFF_VALIDATION_DATA,
            seed,
        )
        models.append(model)
        seed_summaries.append(dict(seed=seed, best_val_r2=history['best_val_r2'],
                                   best_epoch=history['best_epoch'],
                                   stopped_epoch=history['stopped_epoch']))
        score_rows += [
            utils.score_trial_model(model, session, method, 'raw', seed,
                                    query_trials, control_trials)
            for session, (query_trials, control_trials) in STIMULATION_ON_TEST_PAIRS.items()
        ]
        utils.save_checkpoint(model, f'task4_2_{method}_seed{seed}')
        utils.save_training_log(history, method, seed, utils.TRAINING_LOG_DIR)
        print(method, 'seed', seed, 'validation R2', round(history['best_val_r2'], 4))
    return models, pd.DataFrame(score_rows), pd.DataFrame(seed_summaries)


### 4.2.4 Sweeping the kernel length, dropout and depth multiplier

One knob at a time around the default, plus two combinations, one seed each; the best mean
validation R² wins. Since the sweep and the early stopping both look at the validation days,
the selected setting is optimistic *there* — but the test days stay untouched, so the
comparison in 4.2.5 is clean.

In [ ]:
SWEEP_PATH = os.path.join(utils.TASK4_RESULTS, 'eegnet_sweep.csv')
keys = tuple(utils.eegnet_sweep_settings()[0])

# One row per finished setting; a rerun resumes instead of retraining.
sweep_rows = []
if os.path.exists(SWEEP_PATH) and os.path.getsize(SWEEP_PATH):
    sweep_rows = [row for row in pd.read_csv(SWEEP_PATH, on_bad_lines='skip').to_dict('records')
                  if pd.notna(row.get('best_val_r2'))]
    if sweep_rows and not set(keys).issubset(sweep_rows[0]):
        sweep_rows = []  # the settings were renamed since the file was written
seen = {tuple(float(row[key]) for key in keys) for row in sweep_rows}

for setting in utils.eegnet_sweep_settings():
    signature = tuple(float(setting[key]) for key in keys)
    if signature in seen:
        print('already done:', setting)
        continue
    torch.manual_seed(0)  # seed the init as well, not just the shuffling
    _, history = utils.train_with_early_stopping(
        CausalEEGNet(**setting),
        STIMULATION_OFF_TRAIN_DATA,
        STIMULATION_OFF_VALIDATION_DATA,
        0,
    )
    row = dict(**setting, seed=0,
               best_val_r2=history['best_val_r2'],
               best_epoch=history['best_epoch'],
               stopped_epoch=history['stopped_epoch'])
    sweep_rows.append(row)
    seen.add(signature)
    pd.DataFrame([row]).to_csv(
        SWEEP_PATH, mode='a', index=False,
        header=not os.path.exists(SWEEP_PATH) or os.path.getsize(SWEEP_PATH) == 0)
    print(setting, 'validation R2', round(row['best_val_r2'], 4))

SWEEP = pd.DataFrame(sweep_rows).sort_values(
    'best_val_r2', ascending=False).reset_index(drop=True)
SWEEP.to_csv(SWEEP_PATH, index=False)
BEST_SETTINGS = utils.best_eegnet_settings(SWEEP)
print('selected settings:', BEST_SETTINGS)
SWEEP.round(4)


{'temporal_filters_n': 8, 'depth_multiplier': 2, 'kernel_samples': 500, 'separable_samples': 62, 'dropout': 0.5} validation R2 0.1322


### 4.2.5 Five seeds with the best setting

Each seed is early-stopped on the validation days and scored on every stimulated test
session. Five seeds because the seed-to-seed spread is ±0.03 — one seed would not say
anything.

In [ ]:
models, BASELINE_SCORES, SEED_SUMMARY = train_seeds(
    STIMULATION_OFF_TRAIN_DATA, BEST_SETTINGS, utils.EEGNET_METHOD)
SEED_SUMMARY.to_csv(os.path.join(utils.TASK4_RESULTS, 'task4_2_seeds.csv'), index=False)
BASELINE_SCORES.groupby('session')[['r2_off_ave', 'r2_ave']].mean().round(3)

### 4.2.6 Retraining with synthetic stimulation tones

The baseline never saw stimulation, so stimulated input is outside what it was fitted on.
This arm adds 3 Hz and 20 Hz tones to the *training* trials — same architecture, same test
days — to ask whether stimulation-tolerant input helps. Its row sits beside the baseline
and never replaces it.

In [ ]:
FREQ_AUGMENTED_TRAIN_TRIALS = utils.add_stimulation_tones(STIMULATION_OFF_TRAIN_DATA, seed=SEEDS[0])
_, FREQ_SCORES, _ = train_seeds(
    FREQ_AUGMENTED_TRAIN_TRIALS, BEST_SETTINGS, utils.EEGNET_FREQ_AUGMENTATION)
FREQ_SCORES.groupby('session')[['r2_off_ave', 'r2_ave']].mean().round(3)

### 4.2.7 Seed ensemble

Averaging the five seeds' predictions before scoring takes the seed-to-seed spread out of
the comparison. The averaged rows carry `seed = ENSEMBLE_SEED` (-1) so a groupby on `seed`
cannot mistake them for a sixth run of seed 0.

In [ ]:
ENSEMBLE_SCORES = pd.DataFrame([
    utils.score_ensemble(models, session, utils.EEGNET_ENSEMBLE_METHOD, 'raw',
                         query_trials, control_trials, seed=utils.ENSEMBLE_SEED)
    for session, (query_trials, control_trials) in STIMULATION_ON_TEST_PAIRS.items()])

ENSEMBLE_SCORES[['session', 'seed', 'r2_off_ave', 'r2_ave']].round(3)


### 4.2.8 Comparing methods per day

Sessions from one day share a single stimulation-off recording, so they score identically
against it. Averaging per day counts each recording once and leaves five independent
days — the honest unit of comparison.

In [ ]:
ALL_SCORES = pd.concat([BASELINE_SCORES, FREQ_SCORES, ENSEMBLE_SCORES], ignore_index=True)
ALL_SCORES.to_csv(os.path.join(utils.TASK4_RESULTS, 'task4_2_eegnet_off.csv'), index=False)
utils.write_baseline_parquet(ALL_SCORES, os.path.join(utils.RESULTS_DIR, 'results_eegnet.parquet'))

os.makedirs(FIGURE_TRAINING_LOGS, exist_ok=True)
for method in (utils.EEGNET_METHOD, utils.EEGNET_FREQ_AUGMENTATION):
    for seed in SEEDS:
        shutil.copy(os.path.join(utils.TRAINING_LOG_DIR, f'{method}_seed{seed}.json'),
                    FIGURE_TRAINING_LOGS)

print('wrote', len(ALL_SCORES), 'rows for', ALL_SCORES.method.nunique(), 'methods')


In [ ]:
RIDGE = pd.read_csv(os.path.join(utils.RESULTS_DIR, 'task3', 'task3_1_test_results_lmp_bands_lags9.csv'))
RIDGE = (RIDGE.assign(day=RIDGE.session.str.split('-').str[1])
         .groupby('day', as_index=False).r2_off.mean().rename(columns={'r2_off': 'ridge'}))

EEGNET_DAY_SCORES = utils.scores_per_day(ALL_SCORES, 'r2_off_ave')
EEGNET_DAY_SCORES = EEGNET_DAY_SCORES[EEGNET_DAY_SCORES.method == utils.EEGNET_METHOD]
COMPARISON = (EEGNET_DAY_SCORES.rename(columns={'r2_off_ave': 'eegnet'})
              .merge(RIDGE, on='day'))
COMPARISON['eegnet_minus_ridge'] = COMPARISON.eegnet - COMPARISON.ridge
COMPARISON.to_csv(os.path.join(utils.TASK4_RESULTS, 'task4_2_vs_ridge.csv'), index=False)

low, high = utils.bootstrap_confidence_interval(COMPARISON.eegnet_minus_ridge)
print(f'days {len(COMPARISON)} | eegnet {COMPARISON.eegnet.mean():+.3f} | '
      f'ridge {COMPARISON.ridge.mean():+.3f} | difference '
      f'{COMPARISON.eegnet_minus_ridge.mean():+.3f} (95% CI {low:+.3f} to {high:+.3f})')
COMPARISON.round(3)


In [ ]:
# Archived per-session score tables: method -> (file under results/, method label inside it).
ARCHIVED_BASELINE_FILES = {'knn': ('results_ridge_knn.parquet', 'knn_off'),
                           'mlp': ('results_mlp.parquet', 'mlp'),
                           'gru': ('results_gru.parquet', 'gru')}


def archived_baseline_by_day(method):
    """Day-level clean R² for one archived baseline, indexed by recording day."""
    # 1. Open the archived per-session table and keep only this method's rows.
    filename, method_label = ARCHIVED_BASELINE_FILES[method]
    per_session_scores = pd.read_parquet(os.path.join(utils.RESULTS_DIR, filename))
    per_session_scores = per_session_scores[per_session_scores.method == method_label]

    # 2. Session ids look like 'Butters-250520-101021'; field 1 is the recording day.
    per_session_scores = per_session_scores.assign(
        day=per_session_scores.session.str.split('-').str[1])

    # 3. Average the sessions inside each day, then one row per day.
    day_scores = per_session_scores.groupby('day', as_index=False).r2_off_ave.mean()
    return day_scores.rename(columns={'r2_off_ave': method}).set_index('day')


# Day-level table: ridge (task 3.1) first, one column per decoder.
DAY_TABLE = COMPARISON[['day', 'ridge']].set_index('day')

# Task 4 decoders trained here: tidy (method, day) scores -> one column per method.
TASK4_DAY_SCORES = utils.scores_per_day(ALL_SCORES, 'r2_off_ave')
DAY_TABLE = DAY_TABLE.join(TASK4_DAY_SCORES.pivot(index='day', columns='method', values='r2_off_ave'))

# Archived KNN/MLP/GRU baselines, one column each.
for method in ARCHIVED_BASELINE_FILES:
    DAY_TABLE = DAY_TABLE.join(archived_baseline_by_day(method))

DAY_TABLE.reset_index().to_csv(
    os.path.join(utils.TASK4_RESULTS, 'task4_2_baseline_off.csv'), index=False)
DAY_TABLE.round(3)


### 4.2.9 Clean input versus stimulated input

The same checkpoints score well on each test day's stimulation-off recording and poorly on
the stimulated session itself. That gap *is* the domain shift this project is about, so the
two numbers are reported apart rather than averaged into one misleading score.

In [ ]:
CLEAN = utils.day_level_summary(ALL_SCORES, 'r2_off_ave').assign(input='clean')
STIMULATED = utils.day_level_summary(ALL_SCORES, 'r2_ave').assign(input='stimulated')

pd.concat([CLEAN, STIMULATED], ignore_index=True).round(4)


### 4.2.10 Semi-synthetic with 10x artifact

    lfp_10x = lfp_recorded + (scale - 1) * parrm_artifact,   scale = 10

In [ ]:
DATASETS_BY_ARM = utils.amp10_arms(sorted(STIMULATION_ON_TEST_PAIRS))
print(len(STIMULATION_ON_TEST_PAIRS), 'test sessions | arms', list(DATASETS_BY_ARM))

arm_rows = []
for method in (utils.EEGNET_METHOD, utils.EEGNET_FREQ_AUGMENTATION):
    ensemble_models = [utils.load_eegnet_checkpoint(f'task4_2_{method}_seed{seed}')
                       for seed in SEEDS]
    for arm, arm_datasets in DATASETS_BY_ARM.items():
        variant = utils.ARM_VARIANTS[arm]
        for session, (query_trials, control_trials) in arm_datasets.items():
            arm_rows.extend(
                utils.score_trial_model(model, session, method, variant, seed,
                                        query_trials, control_trials)
                for seed, model in enumerate(ensemble_models)
            )
            arm_rows.append(
                utils.score_ensemble(ensemble_models, session, f'{method}_ensemble',
                                     variant, query_trials, control_trials)
            )

ARM_SCORES = pd.DataFrame(arm_rows)
ARM_SCORES.to_csv(os.path.join(utils.TASK4_RESULTS, 'task4_2_eegnet_amp10.csv'), index=False)

(ARM_SCORES.groupby(['method', 'feature_variant'])[['r2_ave', 'r2_off_ave']]
 .mean().unstack().round(4))
